In [1]:
# This code classifies insect sounds in audio files with insects in them.

import librosa
import numpy as np
import pandas as pd
import os
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

In [2]:
# Lists the types of insects

insects = []

taxonomy_df = pd.read_csv("taxonomy.csv")

for row in taxonomy_df.itertuples():
    insects += (row.class_name == 'Insecta')*[row.primary_label]

In [3]:
insects

['1161364',
 '244024',
 '47158son01',
 '47158son02',
 '47158son03',
 '47158son04',
 '47158son05',
 '47158son06',
 '47158son07',
 '47158son08',
 '47158son09',
 '47158son10',
 '47158son11',
 '47158son12',
 '47158son13',
 '47158son14',
 '47158son15',
 '47158son16',
 '47158son17',
 '47158son18',
 '47158son19',
 '47158son20',
 '47158son21',
 '47158son22',
 '47158son23',
 '47158son24',
 '47158son25',
 '760266']

In [4]:
len(insects)

28

In [5]:
train_soundscapes_labels_df = pd.read_csv("train_soundscapes_labels.csv")

train_soundscapes_reduced_df = pd.DataFrame({
    "filename": train_soundscapes_labels_df.iloc[:, :2].astype(str).agg(''.join, axis=1),
    "primary_label": train_soundscapes_labels_df.iloc[:, 3]
})

In [6]:
# Lists all files in train_soundscapes_reduced_df which contain birds.
# Organizes all insects in these files.

train_soundscapes_insects_df = pd.DataFrame(columns = ['filename', 'primary_label'])

for entry in train_soundscapes_reduced_df.itertuples():
    found_insect = False
    primary_labels = entry.primary_label.split(';')
    row_insects = ''
    
    for label in primary_labels:
        if label in insects:
            if found_insect:
                row_insects += ';'
            found_insect = True
            row_insects += label
    
    if found_insect:
        new_row = pd.DataFrame({'filename': [entry.filename], 'primary_label': [row_insects]})
        train_soundscapes_insects_df = pd.concat([train_soundscapes_insects_df, new_row], ignore_index=True)

In [7]:
train_soundscapes_insects_df.head()

,filename,primary_label
0,BC2026_Train_0005_S08_20250607_070007.ogg00:00:00,47158son15;47158son16;47158son17;47158son25
1,BC2026_Train_0005_S08_20250607_070007.ogg00:00:05,47158son15;47158son16;47158son17;47158son25
2,BC2026_Train_0005_S08_20250607_070007.ogg00:00:10,47158son15;47158son16;47158son25
3,BC2026_Train_0005_S08_20250607_070007.ogg00:00:15,47158son15;47158son16;47158son17;47158son25
4,BC2026_Train_0005_S08_20250607_070007.ogg00:00:20,47158son15;47158son16;47158son17;47158son25


In [13]:
# It is very common to have a sound file with multiple insect species.
# 250 out of 336 files in train_soundscapes_insects_df have multiple insects.

multiple_count = 0

for entry in train_soundscapes_insects_df.itertuples():
    if ';' in entry.primary_label:
        multiple_count += 1

print(multiple_count)

250


In [14]:
len(train_soundscapes_insects_df)

336

In [15]:
# In animal_class_classifier.ipynb, we created train_audio_df.csv.
# Each row in this csv corresponds to an audio file in the train_audio folder along with its class.
# Recall that each file in the train_audio folder has exactly one animal in it.
# This loop identifies all bird files in train_audio_df and writes their primary labels.
# Note that the primary label is just the part of the filename that occurs before the '/'.

train_audio_df = pd.read_csv('train_audio_df.csv')

insect_audio_df = pd.DataFrame(columns = ['filename', 'primary_label'])

for entry in train_audio_df.itertuples():
    if entry.primary_label == 'Insecta':
        new_row = pd.DataFrame({'filename': [entry.filename], 'primary_label': [entry.filename.split('/')[0]]})
        insect_audio_df = pd.concat([insect_audio_df, new_row], ignore_index=True)

In [16]:
# In total, we only have 528 files. 336 are in train_soundscapes_insects_df and 192 are in insect_audio_df.
# Crucially, insect_audio_df contains NONE of the sonotypes.

len(insect_audio_df) # 192 files

192

In [17]:
insect_audio_df.head()

,filename,primary_label
0,244024/iNat407846.ogg,244024
1,244024/iNat769302.ogg,244024
2,244024/iNat18148.ogg,244024
3,244024/iNat544406.ogg,244024
4,244024/iNat1232682.ogg,244024


In [18]:
train_soundscapes_insects_df.to_csv('train_soundscapes_insects_df.csv', index=False)
insect_audio_df.to_csv('insect_audio_df.csv', index=False)

In [19]:
# We count the number of files in each class.

counter = {}
for insect in insects:
    counter[insect] = 0

for label in train_soundscapes_insects_df.primary_label:
    labels = label.split(';')
    for primary_label in labels:
        counter[primary_label] += 1

for label in insect_audio_df.primary_label:
    counter[str(label)] += 1

In [20]:
counter.values()

dict_values([10, 176, 46, 14, 66, 34, 6, 36, 96, 34, 12, 66, 72, 10, 72, 24, 24, 24, 86, 24, 10, 24, 44, 48, 48, 48, 168, 6])

In [21]:
# A common class is any class with at least 30 files. The rest of this code is devoted to classifying the common classes.

common_insects = []
rare_insects = []

threshold = 30

for insect in insects:
    if counter[insect] < threshold:
        rare_insects += [insect]
    else:
        common_insects += [insect]

In [22]:
print(common_insects) # There are 16 common insects and 12 rare insects.
print(len(common_insects))
print(rare_insects)
print(len(rare_insects))

['244024', '47158son01', '47158son03', '47158son04', '47158son06', '47158son07', '47158son08', '47158son10', '47158son11', '47158son13', '47158son17', '47158son21', '47158son22', '47158son23', '47158son24', '47158son25']
16
['1161364', '47158son02', '47158son05', '47158son09', '47158son12', '47158son14', '47158son15', '47158son16', '47158son18', '47158son19', '47158son20', '760266']
12


In [23]:
# We reduce train_soundscapes_insects and insect_audio_df into dataframes that only include the common insects.

train_soundscapes_common_insects_df = pd.DataFrame(columns = ['filename', 'primary_label'])

for entry in train_soundscapes_insects_df.itertuples():
    found_common_insect = False
    primary_labels = entry.primary_label.split(';')
    row_common_insects = ''
    
    for label in primary_labels:
        if label in common_insects:
            if found_common_insect:
                row_common_insects += ';'
            found_common_insect = True
            row_common_insects += label
    
    if found_common_insect:
        new_row = pd.DataFrame({'filename': [entry.filename], 'primary_label': [row_common_insects]})
        train_soundscapes_common_insects_df = pd.concat([train_soundscapes_common_insects_df, new_row], ignore_index=True)

In [24]:
# All of the files in train_soundscapes_insects had a common insect.

print(len(train_soundscapes_insects_df), len(train_soundscapes_common_insects_df))

336 336


In [25]:
train_soundscapes_common_insects_df.head()

,filename,primary_label
0,BC2026_Train_0005_S08_20250607_070007.ogg00:00:00,47158son17;47158son25
1,BC2026_Train_0005_S08_20250607_070007.ogg00:00:05,47158son17;47158son25
2,BC2026_Train_0005_S08_20250607_070007.ogg00:00:10,47158son25
3,BC2026_Train_0005_S08_20250607_070007.ogg00:00:15,47158son17;47158son25
4,BC2026_Train_0005_S08_20250607_070007.ogg00:00:20,47158son17;47158son25


In [26]:
common_insect_audio_df = pd.DataFrame(columns = ['filename', 'primary_label'])

for entry in insect_audio_df.itertuples():
    if str(entry.primary_label) in common_insects:
        new_row = pd.DataFrame({'filename': [entry.filename], 'primary_label': [str(entry.primary_label)]})
        common_insect_audio_df = pd.concat([common_insect_audio_df, new_row], ignore_index=True)

In [ ]:
print(len(insect_audio_df), len(common_insect_audio_df)) # 176 out of 192 insect files have a common insect (91.7%)

192 176


In [28]:
common_insect_audio_df.head()

,filename,primary_label
0,244024/iNat407846.ogg,244024
1,244024/iNat769302.ogg,244024
2,244024/iNat18148.ogg,244024
3,244024/iNat544406.ogg,244024
4,244024/iNat1232682.ogg,244024


In [31]:
# We now have all the audio files (train_soundscapes_common_insects_df and common_insect_audio_df).

NUM_CLASSES = len(common_insects)
SR = 32000
N_MELS = 128
X_multi = []
y_multi = []

# This loop converts each audio file in train_soundscapes_common_insects_df to a mel spectrogram.

for row in train_soundscapes_common_insects_df.itertuples():
    filename = row.filename
    labels = row.primary_label.split(';')
    file = filename[:-8]
    start = int(filename[-2:])
    
    audio, _ = librosa.load(
        f"train_soundscapes/{file}",
        sr=SR,
        offset=start,
        duration=5
    )

# Converts the audio to a mel spectrogram, and normalizes it to the range [0, 1].

    mel = librosa.feature.melspectrogram(y=audio, sr=SR, n_mels=N_MELS)
    mel_db = librosa.power_to_db(mel, ref=np.max)

    X_multi.append(mel_db[..., np.newaxis])
    
    # 🔥 MULTI-LABEL TARGET
    label_vector = [0] * NUM_CLASSES
    for i, insect in enumerate(common_insects):
        if insect in labels:
            label_vector[i] = 1

    y_multi.append(label_vector)
    
X_multi = np.array(X_multi, dtype=np.float32)
y_multi = np.array(y_multi, dtype=np.float32)

# X is the list of spectrograms, and y is the list of multi-label vectors.

In [32]:
len(y_multi)

336

In [45]:
# We repeat the same process for the audio files in common_bird_audio_df.
# Because these files are longer than 5 seconds, we split them into 5 second chunks.

X_single = []
y_single = []

EXPECTED_FRAMES = int(np.ceil((SR * 5) / 512))

for row in common_insect_audio_df.itertuples():    
    filename = row.filename
    label = row.primary_label
    
    audio, _ = librosa.load(
        f"train_audio/{filename}",
        sr=SR
    )

    chunk_length = 5 * SR
    
    # In animal_class_classifier, we ensured that all of the ogg files were at least five seconds long.
    audio_clips = []
    for i in range(int(len(audio)/chunk_length)):
        audio_clips += [audio[i*chunk_length:(i + 1)*chunk_length]]

    # Converts the audio files in audio_clips to a mel spectrogram, and normalizes it to the range [0, 1].

    for audio_clip in audio_clips:
        mel = librosa.feature.melspectrogram(
            y=audio_clip,
            sr=SR,
            n_mels=N_MELS
        )

        mel_db = librosa.power_to_db(mel, ref=np.max)

        # Pad / trim
        if mel_db.shape[1] < EXPECTED_FRAMES:
            mel_db = np.pad(mel_db, ((0, 0), (0, EXPECTED_FRAMES - mel_db.shape[1])))
        else:
            mel_db = mel_db[:, :EXPECTED_FRAMES]

        X_single.append(mel_db[..., np.newaxis])

        label_vector = [0] * NUM_CLASSES
        for j, insect in enumerate(common_insects):
            if insect == label:
                label_vector[j] = 1

        y_single.append(label_vector)

X_single = np.array(X_single, dtype=np.float32)
y_single = np.array(y_single, dtype=np.float32)

In [46]:
print(len(y_single)) # We have 336 multi-class files and 970 single class files.

970


In [47]:
X = np.concatenate([X_multi, X_single], axis=0)
y = np.concatenate([y_multi, y_single], axis=0)

In [48]:
len(y)

1306

In [49]:
print((y.sum(axis=0) == 0).sum())

0


In [50]:
print((y_multi.sum(axis=0) == 0).sum()) # 1 class appears in single-class files, but not multi-class files.

1


In [51]:
print((y_single.sum(axis=0) == 0).sum()) # 15 of the classes appear in multi-class files, but not single-class files.

15


In [52]:
# We shuffle the elements of X and y so that we do not over-sample certain types of data.

indices = np.arange(len(X))
np.random.shuffle(indices)

X = X[indices]
y = y[indices]

np.save("X_insect.npy", X)
np.save("y_insect.npy", y)

In [ ]:
# To load the arrays, use this block.

X = np.load("X_insect.npy")
y = np.load("y_insect.npy")

In [ ]:
# Performs a train-test-validation split. We skip SpecAugment because the rest of the code works well without it.

# Step 1: Train vs temp (val + test)

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    shuffle=True
)

# Step 2: Test vs val

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    random_state=42,
    shuffle=True
)

X_train = preprocess(X_train)

In [59]:
print("Train zero classes:", np.sum(y_train.sum(axis=0) == 0))
print("Val zero classes:", np.sum(y_val.sum(axis=0) == 0))
print("Test zero classes:", np.sum(y_test.sum(axis=0) == 0))

print("Train counts:", y_train.sum(axis=0))
print("Val counts:", y_val.sum(axis=0))
print("Test counts:", y_test.sum(axis=0))

Train zero classes: 0
Val zero classes: 0
Test zero classes: 0
Train counts: [690.  30.  46.  24.  24.  63.  25.  42.  52.  46.  57.  26.  29.  29.
  34. 109.]
Val counts: [147.   5.   8.   6.   5.  12.   3.  12.   9.  14.  15.   9.   9.   9.
   7.  26.]
Test counts: [133.  11.  12.   4.   7.  21.   6.  12.  11.  12.  14.   9.  10.  10.
   7.  33.]


In [60]:
# Class 0 (244024) is very overrepresented. In light of this fact, we make two models.
# Model A determines the likelihood that class 0 is present in the audio file.
# Model B determines the likelihood that each of the other 15 classes is present in the audio file.

# Class 0 detector
y_train_0 = y_train[:, 0]
y_val_0   = y_val[:, 0]

# Other-class classifier
X_train_other = X_train
y_train_other = y_train[:, 1:]

X_val_other = X_val
y_val_other = y_val[:, 1:]

In [62]:
# The class0 model determines the likelihood that class 0 is present in the audio file. It is a binary classifier.

def build_class0_model(input_shape=(128, 313, 1)):
    inputs = tf.keras.Input(shape=input_shape)

    x = layers.Conv2D(16, (3,3), padding='same')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D((2,2))(x)

    x = layers.Conv2D(32, (3,3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D((2,2))(x)

    x = layers.Conv2D(64, (3,3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D((2,2))(x)

    x = layers.Conv2D(128, (3,3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.GlobalAveragePooling2D()(x)

    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.3)(x)

    outputs = layers.Dense(1, activation='sigmoid')(x)

    return tf.keras.Model(inputs, outputs)

In [63]:
model_a = build_class0_model()

model_a.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.AUC(curve="ROC", name="roc_auc"),
        tf.keras.metrics.AUC(curve="PR", name="pr_auc"),
        tf.keras.metrics.Precision(),
        tf.keras.metrics.Recall()
    ]
)

In [65]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_pr_auc',
        patience=8,
        min_delta=0.0005,
        mode='max',
        restore_best_weights=True,
        verbose=1
    ),
    
    tf.keras.callbacks.ReduceLROnPlateau( # This block reduces the learning rate near the solution
        monitor='val_loss',
        factor=0.5,
        patience=3,
        cooldown=1,
        mode='min',
        min_lr=1e-6,
        verbose=1
    ),
    
    tf.keras.callbacks.ModelCheckpoint(
        'best_class0_model.keras',
        monitor='val_pr_auc',
        mode='max',
        save_best_only=True,
        verbose=1
    )
]

In [66]:
model_a.fit(
    X_train, y_train_0,
    validation_data=(X_val, y_val_0),
    batch_size=32,
    epochs=30,
    callbacks=callbacks
)

Epoch 1/30
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 206ms/step - loss: 0.4924 - pr_auc: 0.9242 - precision: 0.8325 - recall: 0.9054 - roc_auc: 0.8049
Epoch 1: val_pr_auc improved from None to 0.97563, saving model to best_class0_model.keras

Epoch 1: finished saving model to best_class0_model.keras
29/29 ━━━━━━━━━━━━━━━━━━━━ 8s 224ms/step - loss: 0.4198 - pr_auc: 0.9617 - precision: 0.8902 - recall: 0.9043 - roc_auc: 0.8967 - val_loss: 0.6278 - val_pr_auc: 0.9756 - val_precision: 0.7500 - val_recall: 1.0000 - val_roc_auc: 0.9217 - learning_rate: 1.0000e-04
Epoch 2/30
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 203ms/step - loss: 0.2885 - pr_auc: 0.9830 - precision: 0.9508 - recall: 0.9284 - roc_auc: 0.9612
Epoch 2: val_pr_auc improved from 0.97563 to 0.99222, saving model to best_class0_model.keras

Epoch 2: finished saving model to best_class0_model.keras
29/29 ━━━━━━━━━━━━━━━━━━━━ 6s 213ms/step - loss: 0.2705 - pr_auc: 0.9854 - precision: 0.9639 - recall: 0.9275 - roc_auc: 0.9650 - val_loss: 0.5210 - val_pr_

In [67]:
# The model seems to do well, but here is an additional check.

model_a = tf.keras.models.load_model("best_class0_model.keras")

y_pred_0 = model_a.predict(X_test).ravel()

from sklearn.metrics import classification_report, roc_auc_score, average_precision_score

print("ROC AUC:", roc_auc_score(y_test[:, 0], y_pred_0))
print("PR AUC:", average_precision_score(y_test[:, 0], y_pred_0))

y_bin_0 = (y_pred_0 > 0.5).astype(int)
print(classification_report(y_test[:, 0], y_bin_0))

7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
ROC AUC: 0.9849624060150376
PR AUC: 0.9923143722469345
              precision    recall  f1-score   support

         0.0       0.80      0.95      0.87        63
         1.0       0.98      0.89      0.93       133

    accuracy                           0.91       196
   macro avg       0.89      0.92      0.90       196
weighted avg       0.92      0.91      0.91       196



In [69]:
model_a.save("best_insect0_model.keras")

In [70]:
# We now build model B, which determines the likelihood that each of the other 15 classes is present in the audio file.
# It is a multi-label classifier.

y_train_b = y_train[:, 1:]
y_val_b   = y_val[:, 1:]
y_test_b  = y_test[:, 1:]

NUM_B_CLASSES = y_train_b.shape[1]  # 15

In [71]:
def build_model_b(input_shape=(128, 313, 1), num_classes=15):
    inputs = tf.keras.Input(shape=input_shape)

    x = layers.Conv2D(32, (3,3), padding="same")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D((2,2))(x)

    x = layers.Conv2D(64, (3,3), padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D((2,2))(x)

    x = layers.Conv2D(128, (3,3), padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D((2,2))(x)

    x = layers.Conv2D(256, (3,3), padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    x_avg = layers.GlobalAveragePooling2D()(x)
    x_max = layers.GlobalMaxPooling2D()(x)
    x = layers.Concatenate()([x_avg, x_max])

    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.4)(x)

    outputs = layers.Dense(num_classes, activation="sigmoid")(x)

    return tf.keras.Model(inputs, outputs)

In [72]:
model_b = build_model_b(num_classes=NUM_B_CLASSES)

model_b.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.AUC(curve="ROC", multi_label=True, name="roc_auc"),
        tf.keras.metrics.AUC(curve="PR", multi_label=True, name="pr_auc")
    ]
)

In [73]:
callbacks_b = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_pr_auc",
        patience=8,
        min_delta=0.0005,
        mode="max",
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        cooldown=1,
        mode="min",
        min_lr=1e-6,
        verbose=1
    ),

    tf.keras.callbacks.ModelCheckpoint(
        "best_model_b.keras",
        monitor="val_pr_auc",
        mode="max",
        save_best_only=True,
        verbose=1
    )
]

In [74]:
history_b = model_b.fit(
    X_train, y_train_b,
    validation_data=(X_val, y_val_b),
    batch_size=32,
    epochs=40,
    callbacks=callbacks_b
)

Epoch 1/40
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 556ms/step - loss: 0.6115 - pr_auc: 0.0530 - roc_auc: 0.5550
Epoch 1: val_pr_auc improved from None to 0.19278, saving model to best_model_b.keras

Epoch 1: finished saving model to best_model_b.keras
29/29 ━━━━━━━━━━━━━━━━━━━━ 19s 592ms/step - loss: 0.3776 - pr_auc: 0.0689 - roc_auc: 0.6396 - val_loss: 0.2183 - val_pr_auc: 0.1928 - val_roc_auc: 0.8693 - learning_rate: 1.0000e-04
Epoch 2/40
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 512ms/step - loss: 0.1555 - pr_auc: 0.1635 - roc_auc: 0.8066
Epoch 2: val_pr_auc improved from 0.19278 to 0.21748, saving model to best_model_b.keras

Epoch 2: finished saving model to best_model_b.keras
29/29 ━━━━━━━━━━━━━━━━━━━━ 16s 538ms/step - loss: 0.1537 - pr_auc: 0.1988 - roc_auc: 0.8497 - val_loss: 0.1821 - val_pr_auc: 0.2175 - val_roc_auc: 0.8839 - learning_rate: 1.0000e-04
Epoch 3/40
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 519ms/step - loss: 0.1451 - pr_auc: 0.3250 - roc_auc: 0.8818
Epoch 3: val_pr_auc improved from 0.21748 to 0.

In [77]:
model_b = tf.keras.models.load_model("best_model_b.keras")

y_pred_b = model_b.predict(X_test)

from sklearn.metrics import roc_auc_score, average_precision_score

print("ROC AUC:", roc_auc_score(y_test_b, y_pred_b, average="macro"))
print("PR AUC:", average_precision_score(y_test_b, y_pred_b, average="macro"))

7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 100ms/step
ROC AUC: 0.9805333557588155
PR AUC: 0.9501590523850982


In [78]:
from sklearn.metrics import f1_score
import numpy as np

thresholds_b = []

for i in range(y_test_b.shape[1]):
    best_t = 0.5
    best_f1 = 0

    for t in np.linspace(0.05, 0.9, 50):
        preds = (y_pred_b[:, i] > t).astype(int)
        f1 = f1_score(y_test_b[:, i], preds, zero_division=0)

        if f1 > best_f1:
            best_f1 = f1
            best_t = t

    thresholds_b.append(best_t)
    print(f"Class {i}: threshold={best_t:.3f}, F1={best_f1:.3f}")

Class 0: threshold=0.137, F1=0.900
Class 1: threshold=0.119, F1=0.909
Class 2: threshold=0.206, F1=1.000
Class 3: threshold=0.328, F1=1.000
Class 4: threshold=0.102, F1=1.000
Class 5: threshold=0.067, F1=1.000
Class 6: threshold=0.189, F1=0.960
Class 7: threshold=0.223, F1=0.900
Class 8: threshold=0.085, F1=1.000
Class 9: threshold=0.137, F1=0.966
Class 10: threshold=0.050, F1=0.947
Class 11: threshold=0.067, F1=1.000
Class 12: threshold=0.050, F1=1.000
Class 13: threshold=0.223, F1=0.833
Class 14: threshold=0.137, F1=1.000


In [ ]:
model_b.save("best_insect1_model.keras")